# NanoCore .NET: восстановление конфигурации из RCDATA

Этот блокнот воспроизводит **статическое** преобразование ресурса, найденное вручную в ILSpy. Он не запускает образец и не содержит исполняемого PE-файла. Перед выполнением найдите в коде идентификатор ресурса, `GuidAttribute` сборки и цепочку вызовов, которая задаёт алгоритмы.

**Вход:** только экспортированный `rcdata.bin` (RCDATA №1), полученный в изолированной Windows ВМ. Загрузка файла в Colab передаёт его в облачную среду Google; используйте только разрешённый учебный ресурс. Не загружайте исходный EXE или извлечённые плагины.

Последовательность: установить библиотеку → загрузить ресурс → вписать самостоятельно найденный GUID → расшифровать → разобрать типизированные значения → сопоставить результат с кодом.


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycryptodome"])


In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Загрузите ровно один экспортированный ресурс RCDATA №1")
resource_name, resource = next(iter(uploaded.items()))
if resource[:2] == b"MZ":
    raise ValueError("Загружен PE-файл. Экспортируйте только RCDATA №1")
print("Файл:", resource_name)
print("Размер ресурса:", len(resource), "байт")


In [ ]:
import uuid

# Заполните по GuidAttribute сборки, найденному самостоятельно в ILSpy.
GUID_STR = ""
if not GUID_STR:
    raise ValueError("Сначала впишите GUID сборки в GUID_STR")
guid_bytes = uuid.UUID(GUID_STR).bytes_le  # порядок байтов .NET Guid.ToByteArray()


In [ ]:
import hashlib
import struct
import zlib
from Crypto.Cipher import AES, DES
from Crypto.Util.Padding import unpad

def decrypt_resource(data: bytes, guid: bytes) -> bytes:
    if len(data) < 8:
        raise ValueError("Ресурс слишком короткий")
    size_a = struct.unpack_from("<i", data, 0)[0]
    if size_a <= 0 or 4 + size_a + 4 > len(data):
        raise ValueError("Недопустимая длина блока A")
    block_a = data[4:4 + size_a]
    pos = 4 + size_a
    size_b = struct.unpack_from("<i", data, pos)[0]
    pos += 4
    if size_b <= 0 or pos + size_b > len(data):
        raise ValueError("Недопустимая длина блока B")
    block_b = data[pos:pos + size_b]

    derived = hashlib.pbkdf2_hmac("sha1", guid, guid, 8, dklen=32)
    iv_a, key_a = derived[:16], derived[16:]
    key_des = unpad(AES.new(key_a, AES.MODE_CBC, iv_a).decrypt(block_a), 16)
    if len(key_des) != 8:
        raise ValueError(f"Ожидался DES-ключ 8 байт, получено {len(key_des)}")
    plain = DES.new(key_des, DES.MODE_CBC, key_des).decrypt(block_b)

    compressed = plain[0] != 0
    if compressed:
        expected = struct.unpack_from("<i", plain, 1)[0]
        unpacked = zlib.decompress(plain[5:], -15)
        if len(unpacked) != expected:
            raise ValueError("Длина после Deflate не совпала с заголовком")
        return unpacked
    return plain[1:]

payload = decrypt_resource(resource, guid_bytes)
print("Размер DES-ключа: 8 байт")
print("Размер потока значений:", len(payload), "байт")


In [ ]:
import io

def parse_values(data: bytes):
    stream = io.BytesIO(data)
    def read(n):
        value = stream.read(n)
        if len(value) != n:
            raise ValueError("Неожиданный конец потока значений")
        return value
    def i32(): return struct.unpack("<i", read(4))[0]
    def i64(): return struct.unpack("<q", read(8))[0]
    def read_string():
        length = 0
        shift = 0
        while True:
            byte = read(1)[0]
            length |= (byte & 0x7f) << shift
            if not byte & 0x80: break
            shift += 7
            if shift > 35: raise ValueError("Слишком длинная строка")
        return read(length).decode("utf-8")

    header_a, header_b = read(1)[0], read(1)[0]
    if read(1)[0]:
        header_guid = uuid.UUID(bytes_le=read(16))
    else:
        header_guid = None

    values = []
    while stream.tell() < len(data):
        kind = read(1)[0]
        if kind == 0: value = bool(read(1)[0])
        elif kind == 1: value = read(1)[0]
        elif kind == 2: value = read(i32())
        elif kind == 3: value = read(2).decode("utf-16le")
        elif kind == 7: value = i32()
        elif kind == 8: value = i64()
        elif kind == 12: value = read_string()
        elif kind == 15: value = struct.unpack("<H", read(2))[0]
        elif kind == 16: value = i64()  # DateTime.ToBinary; для конфигурации достаточно сырого значения
        elif kind == 18: value = uuid.UUID(bytes_le=read(16))
        elif kind == 21: value = struct.unpack("<4H", read(8))
        else: raise ValueError(f"Код типа {kind} пока не реализован; позиция {stream.tell() - 1}")
        values.append(value)
    return (header_a, header_b, header_guid), values

header, values = parse_values(payload)
plugin_count = values[0]
plugins = values[1:1 + plugin_count]
setting_count_index = 1 + plugin_count
setting_count = values[setting_count_index]
settings_list = values[setting_count_index + 1:setting_count_index + 1 + setting_count]
if len(settings_list) != setting_count or len(settings_list) % 2:
    raise ValueError("Не сошлось число элементов настроек")
settings = dict(zip(settings_list[::2], settings_list[1::2]))
print("Элементов плагинов:", plugin_count)
print("Ключей настроек:", len(settings))


In [ ]:
for name, value in settings.items():
    shown = f"<массив байтов: {len(value)}>" if isinstance(value, bytes) else repr(value)
    print(f"{name:32} {shown}")

print("\nСтруктура встроенных плагинов (без сохранения исполняемых модулей):")
for index, item in enumerate(plugins):
    if isinstance(item, bytes):
        print(index, "bytes", len(item), "SHA-256", hashlib.sha256(item).hexdigest())
    else:
        print(index, repr(item))


## Проверка результата

Запишите в отчёте: (1) где в декомпилированном коде находится каждый этап преобразования; (2) три поля настроек, подтверждённых выводом; (3) длину DES-ключа; (4) что нельзя заключить о реальной сетевой активности по одному статическому файлу. Приложите собственные скриншоты ILSpy. Порт 443 сам по себе не доказывает использование TLS.

Укажите методы, на которые опирается ваша схема, самостоятельно. Блокнот не выполняет и не сохраняет извлечённые плагины.
